# Notebook 25 — visibility and missingness ablation

**Runtime: T4 GPU.** Press **Run all**. This notebook automatically downloads
the pinned evidence and trains two controlled variants on the exact Notebook 24
split and sampler:

1. V6-C features + 17 MediaPipe visibility channels + 17 binary missing masks.
2. The same representation plus train-only joint/body-part and short-frame
   pose-failure augmentation.

The original V6-C and Notebook 24 data-only reports are fixed controls. No
manual upload is required. The final cell downloads
`calorie_v6d_visibility_missingness_ablation_report.zip`.


In [ ]:
%pip install -q scikit-learn seaborn pandas==2.2.3


In [ ]:
from pathlib import Path
import hashlib
import json
import random
import shutil
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from google.colab import files
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from torch import nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

SEED = 42
WINDOW = 64
STRIDE = 32
BATCH_SIZE = 64
MAX_EPOCHS = 40
PATIENCE = 8
GROUP_TEMPERATURE = 0.5  # frozen from V6-A.1
AUGMENTATION_CONFIG = {
    "horizontal_flip_probability": 0.5,
    "rotation_probability": 0.7,
    "max_rotation_degrees": 7.0,
    "speed_probability": 0.5,
    "speed_range": [0.85, 1.15],
    "frame_drop_probability": 0.35,
    "frame_drop_fraction_range": [0.05, 0.12],
    "coordinate_noise_probability": 0.8,
    "coordinate_noise_std": 0.008,
}
RETIRED_CLASSES = {"plank"}
NEW_TRAIN_GROUPS = {"youtube:Fk1wnNX4wQ8", "youtube:zxqCV_eQHZY"}
UNREVIEWED_GROUPS = {
    "youtube:oLsonHQJOUc",
    "youtube:c4TomZjzobY",
    "youtube:W2Bm9dfQjbk",
}


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("Experiment: visibility/missingness controlled ablation")
assert DEVICE.type == "cuda", "Select a T4 runtime before training."


## Download pinned training evidence and both controls

Inputs come from immutable commit `7e7a586`. Critical files are SHA-256 checked
and every archive is structurally tested before extraction.


In [ ]:
PINNED_COMMIT = "7e7a586d0f0294711c8c6f9cd3f79cfaa7835c6e"
REPOSITORY_RAW = (
    "https://raw.githubusercontent.com/bariscelikk1/cALorie/"
    f"{PINNED_COMMIT}/ml/artifacts"
)
LANDMARK_ARCHIVES = [
    ("v1/landmarks", "calorie_workoutfitness_landmarks_v1.zip"),
    ("v1/landmarks", "calorie_mmfit_landmarks_v1.zip"),
    ("v1/landmarks", "calorie_mmfit_unknown_v1.zip"),
    ("v2/landmarks", "calorie_youtube_mixed_landmarks_v3.zip"),
    ("v3/landmarks", "calorie_youtube_additional_landmarks_v1.zip"),
    ("v3/landmarks", "calorie_youtube_training_landmarks_v4.zip"),
    ("v5/landmarks", "calorie_workoutfitness_gate_hardneg_v1.zip"),
    ("youtube_long_v2/reviewed_clips", "calorie_youtube_long_workouts_v2_reviewed_clips.zip"),
]
EXPECTED_HASHES = {
    "calorie_youtube_long_workouts_v2_reviewed_clips.zip": "84fb55b981820af702a94f0c0070960b24fdd1acb617097f1838c782cd942284",
    "hard_positive_windows.csv": "5a4b73c04ee85659570e7931ea0e84b35c0abeb16fd3f08238e629f6eca091c5",
    "split_assignments.csv": "8c013695fe095ceccc4ff65722d2a26e5a309674f5b877b840b71ba73dc085d6",
    "baseline_metrics.json": "baee2ea099402ca25b0e2bc322809dfa7a0f8d1e9969204792cd5d60bfb96b2b",
    "baseline_test_predictions.csv": "b34b6cada90355682ef7aa225e3d18dad0960dd79ce32856ebb524356734326c",
    "data_only_metrics.json": "cdc3d944063d12226a2ceaab6be9afd4e4cc23b6031dddc52a6ad86bb43c0670",
    "data_only_test_predictions.csv": "b379c30e563ea626d1f411c3ac372280820db03e7bd86dded647bc90a14adc65",
}


def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def download(url, destination, expected=None):
    if destination.exists() and (expected is None or sha256(destination) == expected):
        print("Cached:", destination.name)
        return
    temporary = destination.with_suffix(destination.suffix + ".part")
    temporary.unlink(missing_ok=True)
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=300) as response, temporary.open("wb") as output:
        shutil.copyfileobj(response, output)
    if expected is not None:
        actual = sha256(temporary)
        assert actual == expected, f"SHA-256 mismatch for {destination.name}: {actual}"
    temporary.replace(destination)
    print(f"{destination.name}: {destination.stat().st_size / 1e6:.1f} MB")


zip_paths = []
for relative_root, name in LANDMARK_ARCHIVES:
    destination = Path("/content") / name
    download(
        f"{REPOSITORY_RAW}/{relative_root}/{name}", destination,
        EXPECTED_HASHES.get(name),
    )
    with zipfile.ZipFile(destination) as archive:
        assert archive.testzip() is None, f"Corrupt ZIP: {name}"
    zip_paths.append(destination)

HARD_POSITIVE_PATH = Path("/content/hard_positive_windows.csv")
download(
    f"{REPOSITORY_RAW}/v5/hard_positive_mining/hard_positive_windows.csv",
    HARD_POSITIVE_PATH, EXPECTED_HASHES["hard_positive_windows.csv"],
)
hard_positive_frame = pd.read_csv(HARD_POSITIVE_PATH)
assert hard_positive_frame.window_key.is_unique

V6_SPLIT_PATH = Path("/content/split_assignments.csv")
download(
    f"{REPOSITORY_RAW}/v6a1/model/split_assignments.csv",
    V6_SPLIT_PATH, EXPECTED_HASHES["split_assignments.csv"],
)
BASELINE_METRICS_PATH = Path("/content/baseline_metrics.json")
download(
    f"{REPOSITORY_RAW}/v6c/model/metrics.json",
    BASELINE_METRICS_PATH, EXPECTED_HASHES["baseline_metrics.json"],
)
BASELINE_PREDICTIONS_PATH = Path("/content/baseline_test_predictions.csv")
download(
    f"{REPOSITORY_RAW}/v6c/model/test_window_predictions.csv",
    BASELINE_PREDICTIONS_PATH, EXPECTED_HASHES["baseline_test_predictions.csv"],
)
baseline_metrics = json.loads(BASELINE_METRICS_PATH.read_text())
baseline_predictions = pd.read_csv(BASELINE_PREDICTIONS_PATH)
DATA_ONLY_METRICS_PATH = Path("/content/data_only_metrics.json")
download(
    f"{REPOSITORY_RAW}/v6c_longv2_data_retrain/model/metrics.json",
    DATA_ONLY_METRICS_PATH, EXPECTED_HASHES["data_only_metrics.json"],
)
DATA_ONLY_PREDICTIONS_PATH = Path("/content/data_only_test_predictions.csv")
download(
    f"{REPOSITORY_RAW}/v6c_longv2_data_retrain/model/test_window_predictions.csv",
    DATA_ONLY_PREDICTIONS_PATH,
    EXPECTED_HASHES["data_only_test_predictions.csv"],
)
data_only_metrics = json.loads(DATA_ONLY_METRICS_PATH.read_text())
data_only_predictions = pd.read_csv(DATA_ONLY_PREDICTIONS_PATH)

SOURCE_ROOT = Path("/content/v6c_longv2_sources")
if SOURCE_ROOT.exists():
    shutil.rmtree(SOURCE_ROOT)
SOURCE_ROOT.mkdir()
for index, archive_path in enumerate(zip_paths):
    destination = SOURCE_ROOT / f"source_{index}"
    destination.mkdir()
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(destination)

frames = []
for manifest_path in SOURCE_ROOT.rglob("manifest.csv"):
    frame = pd.read_csv(manifest_path).fillna("")
    frame["npz_path"] = frame["output_path"].map(
        lambda value: str(manifest_path.parent / value)
    )
    frames.append(frame)
assert frames, "No manifest.csv was found."
manifest = pd.concat(frames, ignore_index=True)
manifest = manifest[
    (manifest.status == "ok")
    & manifest.npz_path.map(lambda value: Path(value).exists())
].copy()
if "group_id" not in manifest or manifest.group_id.eq("").any():
    manifest["group_id"] = manifest.apply(
        lambda row: f"{row['source_dataset']}:{row['clip_id']}", axis=1
    )
assert NEW_TRAIN_GROUPS <= set(manifest.group_id)
print("Valid clips:", len(manifest), "Groups:", manifest.group_id.nunique())
display(manifest.canonical_class.value_counts().sort_index().rename("clips"))


## Rebuild the frozen split and audit visibility

The 1,971 test keys and labels must match both original V6-C and Notebook 24.
Visibility statistics are reported by source because MM-Fit uses synthetic
visibility of one while MediaPipe sources contain measured confidence.


In [ ]:
def resample_sequence(sequence, source_fps, target_fps=15):
    if not source_fps or abs(float(source_fps) - target_fps) < 0.1:
        return sequence
    duration = (len(sequence) - 1) / float(source_fps)
    target_length = max(2, int(round(duration * target_fps)) + 1)
    old_x = np.linspace(0, duration, len(sequence))
    new_x = np.linspace(0, duration, target_length)
    flat = sequence.reshape(len(sequence), -1)
    sampled = np.stack([
        np.interp(new_x, old_x, flat[:, index])
        for index in range(flat.shape[1])
    ], axis=1)
    return sampled.reshape(target_length, 17, 4).astype(np.float32)


windows, original_labels, groups = [], [], []
clip_ids, sources, starts = [], [], []
for row in manifest.itertuples(index=False):
    with np.load(row.npz_path, allow_pickle=False) as item:
        sequence = item["landmarks"].astype(np.float32)
    if sequence.ndim != 3 or sequence.shape[1:] != (17, 4):
        print("Skipping unexpected shape:", row.npz_path, sequence.shape)
        continue
    sequence = resample_sequence(sequence, getattr(row, "source_fps", 15) or 15)
    clip_starts = list(range(0, max(1, len(sequence) - WINDOW + 1), STRIDE))
    if len(sequence) > WINDOW and clip_starts[-1] != len(sequence) - WINDOW:
        clip_starts.append(len(sequence) - WINDOW)
    for start in clip_starts:
        window = sequence[start:start + WINDOW]
        if len(window) < WINDOW:
            window = np.concatenate([
                window, np.repeat(window[-1:], WINDOW - len(window), axis=0)
            ])
        windows.append(window)
        original_labels.append(row.canonical_class)
        groups.append(row.group_id)
        clip_ids.append(row.clip_id)
        sources.append(row.source_dataset)
        starts.append(start)

X = np.asarray(windows, dtype=np.float32)
original_labels = np.asarray(original_labels)
groups = np.asarray(groups)
clip_ids = np.asarray(clip_ids)
sources = np.asarray(sources)
starts = np.asarray(starts, dtype=np.int64)
window_keys = np.asarray([
    f"{source}|{group}|{clip}|{start}"
    for source, group, clip, start in zip(sources, groups, clip_ids, starts)
])

# Reuse the exact group assignment exported by V6-A.1.
frozen_split = pd.read_csv(V6_SPLIT_PATH)
assert frozen_split.groupby("group_id")["split"].nunique().max() == 1
group_to_split = (
    frozen_split[["group_id", "split"]]
    .drop_duplicates("group_id")
    .set_index("group_id")["split"]
    .to_dict()
)
old_evidence_mask = np.asarray([group in group_to_split for group in groups])
assigned_split = np.asarray([group_to_split.get(group, "") for group in groups])
new_train_mask = np.isin(groups, list(NEW_TRAIN_GROUPS))
assigned_split[new_train_mask] = "train"
missing_groups = set(groups[assigned_split == ""])
assert missing_groups <= UNREVIEWED_GROUPS, (
    "Unexpected unresolved groups: " + str(sorted(missing_groups))
)
assert set(groups[new_train_mask]) == NEW_TRAIN_GROUPS
train_idx = np.flatnonzero(assigned_split == "train")
val_idx = np.flatnonzero(assigned_split == "validation")
test_idx = np.flatnonzero(assigned_split == "test")

labels = np.where(np.isin(original_labels, list(RETIRED_CLASSES)), "unknown", original_labels)
ALL_CLASSES = sorted(np.unique(labels).tolist())
SUPPORTED_CLASSES = [name for name in ALL_CLASSES if name != "unknown"]
assert len(SUPPORTED_CLASSES) == 14, SUPPORTED_CLASSES
all_to_id = {name: index for index, name in enumerate(ALL_CLASSES)}
supported_to_id = {name: index for index, name in enumerate(SUPPORTED_CLASSES)}
y_all = np.asarray([all_to_id[name] for name in labels], dtype=np.int64)
y_gate = (labels != "unknown").astype(np.int64)
y_exercise = np.asarray([supported_to_id.get(name, -1) for name in labels], dtype=np.int64)

assert not (set(groups[train_idx]) & set(groups[val_idx]))
assert not (set(groups[train_idx]) & set(groups[test_idx]))
assert not (set(groups[val_idx]) & set(groups[test_idx]))
assert not (set(groups[train_idx]) & UNREVIEWED_GROUPS)
assert set(window_keys[old_evidence_mask]) == set(frozen_split.window_key), (
    "Reviewed window evidence changed since V6-A.1."
)
assert set(window_keys[test_idx]) == set(baseline_predictions.window_key)
assert len(test_idx) == 1971, len(test_idx)
assert set(window_keys[test_idx]) == set(data_only_predictions.window_key)
baseline_truth_by_key = dict(zip(baseline_predictions.window_key, baseline_predictions.true_class))
assert all(baseline_truth_by_key[key] == truth for key, truth in zip(window_keys[test_idx], labels[test_idx])), "Frozen test labels changed relative to V6-C."
assert not set(groups[val_idx]) & NEW_TRAIN_GROUPS
assert not set(groups[test_idx]) & NEW_TRAIN_GROUPS
print("Excluded unresolved groups:", sorted(missing_groups))
print("New train-only groups:", sorted(NEW_TRAIN_GROUPS))

print("Windows:", X.shape)
print("Frozen groups:", len(group_to_split))
print("Active supported classes:", SUPPORTED_CLASSES)
print("Retired plank windows mapped to unknown:", int((original_labels == "plank").sum()))
for name, indices in [("train", train_idx), ("validation", val_idx), ("test", test_idx)]:
    print(name, "windows=", len(indices), "groups=", len(np.unique(groups[indices])))
    display(pd.Series(labels[indices]).value_counts().sort_index().rename("windows"))


new_indices = np.flatnonzero(new_train_mask)
data_delta_audit = pd.DataFrame({
    "canonical_class": labels[new_indices],
    "group_id": groups[new_indices],
    "window_key": window_keys[new_indices],
}).groupby(["canonical_class", "group_id"]).agg(
    windows=("window_key", "count")
).reset_index()
class_creator_support_matrix = pd.crosstab(
    pd.Series(labels[train_idx], name="canonical_class"),
    pd.Series(groups[train_idx], name="group_id"),
)
print("Added training windows:", len(new_indices))
display(data_delta_audit)
display(class_creator_support_matrix)


visibility_values = np.clip(X[:, :, :, 3], 0.0, 1.0)
visibility_audit = pd.DataFrame({
    "source_dataset": sources,
    "split": assigned_split,
    "mean_visibility": visibility_values.mean(axis=(1, 2)),
    "missing_fraction": (visibility_values < 0.5).mean(axis=(1, 2)),
})
visibility_audit = visibility_audit[visibility_audit.split != ""].groupby(
    ["source_dataset", "split"]
).agg(
    windows=("mean_visibility", "count"),
    mean_visibility=("mean_visibility", "mean"),
    missing_fraction=("missing_fraction", "mean"),
).reset_index()
display(visibility_audit.round(4))


## Tempered hierarchical sampling and hard-positive weighting

Sampling first assigns equal probability to each class. Inside a class, each
source group receives probability proportional to `window_count ** 0.5`. This is
the midpoint between failed V6-A equal-group sampling (`** 0`) and raw window
sampling (`** 1`): long videos are restrained without making a one-window group
as influential as a large group. Notebook 13 hard-positive modifiers are
normalized inside the same class/group, so they change which windows are seen
without changing that group's total probability.

For the gate, “class” means supported or unknown. For the exercise model it
means one of the 14 active movements. `WeightedRandomSampler` then draws the
requested class -> group -> window distribution.


In [ ]:
hard_weight_by_key = dict(zip(
    hard_positive_frame.window_key,
    hard_positive_frame.recommended_sample_weight,
))
hard_positive_weights = np.asarray([
    float(hard_weight_by_key.get(key, 1.0)) for key in window_keys
], dtype=np.float64)
# Retired plank is now unknown and cannot receive a supported hard-positive boost.
hard_positive_weights[labels == "unknown"] = 1.0
matched_training_keys = (
    set(window_keys[train_idx][labels[train_idx] != "unknown"])
    & set(hard_weight_by_key)
)
assert not (set(window_keys[val_idx]) & set(hard_weight_by_key)), (
    "Notebook 13 weights unexpectedly intersect validation."
)
assert not (set(window_keys[test_idx]) & set(hard_weight_by_key)), (
    "Notebook 13 weights unexpectedly intersect test."
)
print("Notebook 13 keys:", len(hard_weight_by_key))
print("Matched active training keys:", len(matched_training_keys))
print("Upweighted active training windows:", int((hard_positive_weights[train_idx] > 1).sum()))


def hierarchical_window_weights(indices, hierarchy_labels, extra_weights=None):
    table = pd.DataFrame({
        "index": indices,
        "hierarchy_label": hierarchy_labels[indices],
        "group": groups[indices],
    })
    class_count = table["hierarchy_label"].nunique()
    group_sizes = table.groupby(
        ["hierarchy_label", "group"]
    )["index"].transform("count").astype(float)
    table["group_mass"] = group_sizes.pow(GROUP_TEMPERATURE)
    group_table = table[["hierarchy_label", "group"]].copy()
    group_table["group_mass"] = table["group_mass"]
    class_group_mass = (
        group_table.drop_duplicates(["hierarchy_label", "group"])
        .groupby("hierarchy_label")["group_mass"].sum()
    )
    table["extra_weight"] = (
        1.0 if extra_weights is None else extra_weights[table["index"].to_numpy()]
    )
    table["pair_extra_total"] = table.groupby(
        ["hierarchy_label", "group"]
    )["extra_weight"].transform("sum")
    weights = []
    for row in table.itertuples(index=False):
        weight = (
            1.0 / class_count
            * row.group_mass / class_group_mass.loc[row.hierarchy_label]
            * row.extra_weight / row.pair_extra_total
        )
        weights.append(weight)
    weights = np.asarray(weights, dtype=np.float64)
    return weights / weights.mean()


# Use strings here so the audit table remains human-readable.
gate_hierarchy = np.where(y_gate == 1, "supported", "unknown")
gate_train_weights = hierarchical_window_weights(
    train_idx, gate_hierarchy, hard_positive_weights
)
exercise_train_idx = train_idx[y_exercise[train_idx] >= 0]
exercise_val_idx = val_idx[y_exercise[val_idx] >= 0]
exercise_test_idx = test_idx[y_exercise[test_idx] >= 0]
exercise_train_weights = hierarchical_window_weights(
    exercise_train_idx, labels
)

sampler_audit = pd.DataFrame({
    "window_key": window_keys[train_idx],
    "canonical_class": labels[train_idx],
    "original_class": original_labels[train_idx],
    "group_id": groups[train_idx],
    "gate_target": gate_hierarchy[train_idx],
    "hard_positive_weight": hard_positive_weights[train_idx],
    "gate_sampler_weight": gate_train_weights,
})
exercise_weight_map = dict(zip(
    window_keys[exercise_train_idx], exercise_train_weights
))
sampler_audit["exercise_sampler_weight"] = sampler_audit.window_key.map(
    exercise_weight_map
).fillna(0.0)

display(sampler_audit.groupby("gate_target").agg(
    windows=("window_key", "count"),
    groups=("group_id", "nunique"),
    total_sampler_weight=("gate_sampler_weight", "sum"),
).round(3))
display(sampler_audit[sampler_audit.canonical_class != "unknown"].groupby(
    "canonical_class"
).agg(
    windows=("window_key", "count"),
    groups=("group_id", "nunique"),
    total_exercise_weight=("exercise_sampler_weight", "sum"),
).round(3))


In [ ]:
LEFT_RIGHT_PAIRS = [
    (1, 2), (3, 4), (5, 6), (7, 8),
    (9, 10), (11, 12), (13, 14), (15, 16),
]
BODY_PARTS = {
    "left_arm": [5, 7, 9], "right_arm": [6, 8, 10],
    "left_leg": [11, 13, 15], "right_leg": [12, 14, 16],
    "upper_body": [5, 6, 7, 8, 9, 10],
}
OCCLUSION_CONFIG = {
    "body_part_probability": 0.35,
    "body_part_span_frames": [4, 12],
    "frame_occlusion_probability": 0.25,
    "frame_span_frames": [2, 6],
    "low_visibility_noise_std": 0.012,
}


def augment_pose_window(sample, force_all=False):
    sample = sample.copy()
    config = AUGMENTATION_CONFIG
    if force_all or np.random.random() < config["horizontal_flip_probability"]:
        sample[:, :, 0] *= -1.0
        for left, right in LEFT_RIGHT_PAIRS:
            sample[:, [left, right], :] = sample[:, [right, left], :]
    if force_all or np.random.random() < config["rotation_probability"]:
        degrees = np.random.uniform(-config["max_rotation_degrees"], config["max_rotation_degrees"])
        radians = np.deg2rad(degrees)
        cosine, sine = np.cos(radians), np.sin(radians)
        x, y = sample[:, :, 0].copy(), sample[:, :, 1].copy()
        sample[:, :, 0] = cosine * x - sine * y
        sample[:, :, 1] = sine * x + cosine * y
    if force_all or np.random.random() < config["speed_probability"]:
        speed = np.random.uniform(*config["speed_range"])
        center = (WINDOW - 1) / 2.0
        positions = (np.arange(WINDOW) - center) * speed + center
        flat = sample.reshape(WINDOW, -1)
        flat = np.stack([np.interp(positions, np.arange(WINDOW), flat[:, c]) for c in range(flat.shape[1])], axis=1)
        sample = flat.reshape(WINDOW, 17, 4).astype(np.float32)
    if force_all or np.random.random() < config["frame_drop_probability"]:
        fraction = np.random.uniform(*config["frame_drop_fraction_range"])
        count = max(1, int(round(WINDOW * fraction)))
        dropped = np.random.choice(np.arange(1, WINDOW), size=count, replace=False)
        sample[dropped] = sample[dropped - 1]
    if force_all or np.random.random() < config["coordinate_noise_probability"]:
        sample[:, :, :2] += np.random.normal(0, config["coordinate_noise_std"], sample[:, :, :2].shape).astype(np.float32)
    return sample.astype(np.float32)


def realistic_pose_failure(sample, rng, force_all=False):
    sample = sample.copy()
    if force_all or rng.random() < OCCLUSION_CONFIG["body_part_probability"]:
        joints = np.asarray(BODY_PARTS[rng.choice(list(BODY_PARTS))])
        length = int(rng.integers(OCCLUSION_CONFIG["body_part_span_frames"][0], OCCLUSION_CONFIG["body_part_span_frames"][1] + 1))
        start = int(rng.integers(1, WINDOW - length + 1))
        sample[start:start + length, joints, :3] = sample[start - 1, joints, :3]
        sample[start:start + length, joints, 3] = 0.0
    if force_all or rng.random() < OCCLUSION_CONFIG["frame_occlusion_probability"]:
        length = int(rng.integers(OCCLUSION_CONFIG["frame_span_frames"][0], OCCLUSION_CONFIG["frame_span_frames"][1] + 1))
        start = int(rng.integers(1, WINDOW - length + 1))
        sample[start:start + length, :, :3] = sample[start - 1, :, :3]
        sample[start:start + length, :, 3] = 0.0
    visibility = np.clip(sample[:, :, 3:4], 0.0, 1.0)
    noise_scale = (1.0 - visibility) * OCCLUSION_CONFIG["low_visibility_noise_std"]
    sample[:, :, :2] += rng.normal(0.0, 1.0, sample[:, :, :2].shape).astype(np.float32) * noise_scale
    return sample.astype(np.float32)


BONE_EDGES = [(5, 6), (5, 7), (7, 9), (6, 8), (8, 10), (5, 11), (6, 12), (11, 12), (11, 13), (13, 15), (12, 14), (14, 16)]
ANGLE_TRIPLETS = [(5, 7, 9), (6, 8, 10), (7, 5, 11), (8, 6, 12), (5, 11, 13), (6, 12, 14), (11, 13, 15), (12, 14, 16)]
BASE_FEATURE_CONFIG = {"coordinates_xy": 34, "velocity_xy": 34, "bone_vectors_xy": 24, "joint_angles": 8}
MASK_FEATURE_CONFIG = {**BASE_FEATURE_CONFIG, "visibility": 17, "missing_mask": 17}


def pose_to_features(sample, mode="visibility_mask"):
    xy = sample[:, :, :2].astype(np.float32)
    velocity = np.zeros_like(xy); velocity[1:] = xy[1:] - xy[:-1]
    bones = np.stack([xy[:, child] - xy[:, parent] for parent, child in BONE_EDGES], axis=1)
    angles = []
    for first, center, last in ANGLE_TRIPLETS:
        a, b = xy[:, first] - xy[:, center], xy[:, last] - xy[:, center]
        cosine = np.sum(a * b, axis=1) / np.maximum(np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1), 1e-6)
        angles.append(np.arccos(np.clip(cosine, -1.0, 1.0)) / np.pi)
    blocks = [xy.reshape(WINDOW, -1), velocity.reshape(WINDOW, -1), bones.reshape(WINDOW, -1), np.stack(angles, axis=1)]
    if mode == "visibility_mask":
        visibility = np.clip(sample[:, :, 3], 0.0, 1.0)
        blocks.extend([visibility, (visibility < 0.5).astype(np.float32)])
    features = np.concatenate(blocks, axis=1).astype(np.float32)
    assert np.isfinite(features).all()
    return features


class PoseWindowDataset(Dataset):
    def __init__(self, indices, targets, training, mode, realistic_occlusion, occlusion_seed):
        self.indices, self.targets, self.training = np.asarray(indices), targets, training
        self.mode, self.realistic_occlusion = mode, realistic_occlusion
        self.occlusion_rng = np.random.default_rng(occlusion_seed)
    def __len__(self): return len(self.indices)
    def __getitem__(self, item):
        actual = self.indices[item]
        sample = X[actual].copy()
        if self.training:
            sample = augment_pose_window(sample)
            if self.realistic_occlusion:
                sample = realistic_pose_failure(sample, self.occlusion_rng)
        return torch.from_numpy(pose_to_features(sample, self.mode).T), torch.tensor(self.targets[actual], dtype=torch.long)


def make_loaders(targets, train_indices, val_indices, test_indices, train_weights, seed, mode, realistic_occlusion):
    generator = torch.Generator().manual_seed(seed)
    sampler = WeightedRandomSampler(torch.as_tensor(train_weights, dtype=torch.double), len(train_weights), replacement=True, generator=generator)
    make = lambda idx, training: PoseWindowDataset(idx, targets, training, mode, realistic_occlusion and training, seed + 9000)
    return (
        DataLoader(make(train_indices, True), batch_size=BATCH_SIZE, sampler=sampler),
        DataLoader(make(val_indices, False), batch_size=BATCH_SIZE, shuffle=False),
        DataLoader(make(test_indices, False), batch_size=BATCH_SIZE, shuffle=False),
    )


VARIANTS = {
    "visibility_mask": {"mode": "visibility_mask", "realistic_occlusion": False, "input_features": 134},
    "visibility_mask_occlusion": {"mode": "visibility_mask", "realistic_occlusion": True, "input_features": 134},
}
print(VARIANTS)


## Visual audit of missingness augmentation

The third row shows train-only body-part and frame loss. Color encodes
visibility; missing joints are held at the last observed position and marked
with zero visibility rather than replaced with implausible coordinates.


In [ ]:
preview_candidates = train_idx[labels[train_idx] == "squat"]
preview_index = int(preview_candidates[0] if len(preview_candidates) else train_idx[0])
original_preview = X[preview_index].copy()
np_state = np.random.get_state(); np.random.seed(SEED + 1000)
base_augmented = augment_pose_window(original_preview, force_all=True)
np.random.set_state(np_state)
occluded_preview = realistic_pose_failure(base_augmented, np.random.default_rng(SEED + 2000), force_all=True)
fig, axes = plt.subplots(3, 6, figsize=(15, 9))
for row, (title, sequence) in enumerate([("original", original_preview), ("frozen augmentation", base_augmented), ("pose-failure augmentation", occluded_preview)]):
    for axis, frame_index in zip(axes[row], np.linspace(0, WINDOW - 1, 6, dtype=int)):
        xy, visibility = sequence[frame_index, :, :2], np.clip(sequence[frame_index, :, 3], 0, 1)
        for left, right in BONE_EDGES:
            axis.plot([xy[left, 0], xy[right, 0]], [-xy[left, 1], -xy[right, 1]], color="tab:blue", linewidth=1.4, alpha=float(min(visibility[left], visibility[right])))
        axis.scatter(xy[:, 0], -xy[:, 1], s=14, c=visibility, cmap="RdYlGn", vmin=0, vmax=1)
        axis.set_aspect("equal"); axis.axis("off"); axis.set_title(f"{title} t={frame_index}")
fig.suptitle(f"Notebook 25 missingness audit | {labels[preview_index]}")
fig.tight_layout()
AUGMENTATION_PREVIEW_PATH = Path("/content/v6d_missingness_preview.png")
fig.savefig(AUGMENTATION_PREVIEW_PATH, dpi=170, bbox_inches="tight")
plt.show()


## Frozen TCN, two controlled 134-channel variants

Both variants use the original V6-C TCN. The second differs from the first only
by train-only realistic pose-failure augmentation.


In [ ]:
class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dilation, dropout=0.2):
        super().__init__()
        padding = dilation
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, 3, padding=padding, dilation=dilation),
            nn.BatchNorm1d(out_channels), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(out_channels, out_channels, 3, padding=padding, dilation=dilation),
            nn.BatchNorm1d(out_channels), nn.ReLU(), nn.Dropout(dropout),
        )
        self.residual = (
            nn.Conv1d(in_channels, out_channels, 1)
            if in_channels != out_channels else nn.Identity()
        )

    def forward(self, x):
        return torch.relu(self.net(x) + self.residual(x))


class SmallTCN(nn.Module):
    def __init__(self, input_features, num_classes):
        super().__init__()
        channels = [64, 96, 128, 128]
        blocks, current = [], input_features
        for level, output in enumerate(channels):
            blocks.append(TemporalBlock(current, output, dilation=2 ** level))
            current = output
        self.features = nn.Sequential(*blocks)
        self.classifier = nn.Linear(current, num_classes)

    def forward(self, x):
        return self.classifier(self.features(x).mean(dim=-1))


In [ ]:
def predict_model(model, loader):
    model.eval(); truths, probabilities = [], []
    with torch.no_grad():
        for features, targets in loader:
            logits = model(features.to(DEVICE)); truths.extend(targets.numpy()); probabilities.extend(logits.softmax(1).cpu().numpy())
    probabilities = np.asarray(probabilities)
    return np.asarray(truths), probabilities.argmax(1), probabilities


def train_model(name, num_classes, loaders, seed, input_features):
    seed_everything(seed)
    model = SmallTCN(input_features, num_classes).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", patience=3, factor=0.5)
    best_path = Path(f"/content/best_v6d_{name}.pt")
    best_f1, stale, history = -1.0, 0, []
    def run_epoch(loader, training):
        model.train(training); losses, truths, predictions = [], [], []
        with (torch.enable_grad() if training else torch.no_grad()):
            for features, targets in loader:
                features, targets = features.to(DEVICE), targets.to(DEVICE)
                if training: optimizer.zero_grad(set_to_none=True)
                logits = model(features); loss = criterion(logits, targets)
                if training: loss.backward(); optimizer.step()
                losses.append(loss.item() * len(targets)); truths.extend(targets.cpu().numpy()); predictions.extend(logits.argmax(1).cpu().numpy())
        return sum(losses) / len(truths), f1_score(truths, predictions, average="macro", zero_division=0)
    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss, train_f1 = run_epoch(loaders[0], True); val_loss, val_f1 = run_epoch(loaders[1], False)
        scheduler.step(val_f1); history.append({"model": name, "epoch": epoch, "train_loss": train_loss, "train_macro_f1": train_f1, "val_loss": val_loss, "val_macro_f1": val_f1})
        print(f"{name} {epoch:02d} train F1={train_f1:.3f} val F1={val_f1:.3f}")
        if val_f1 > best_f1 + 1e-4: best_f1, stale = val_f1, 0; torch.save(model.state_dict(), best_path)
        else:
            stale += 1
            if stale >= PATIENCE: break
    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    return model, history, best_f1


results = {}
for variant, config in VARIANTS.items():
    print("TRAINING", variant)
    gate_loaders = make_loaders(y_gate, train_idx, val_idx, test_idx, gate_train_weights, SEED, config["mode"], config["realistic_occlusion"])
    exercise_loaders = make_loaders(y_exercise, exercise_train_idx, exercise_val_idx, exercise_test_idx, exercise_train_weights, SEED + 1, config["mode"], config["realistic_occlusion"])
    gate_model, gate_history, gate_best = train_model(f"{variant}_gate", 2, gate_loaders, SEED, config["input_features"])
    exercise_model, exercise_history, exercise_best = train_model(f"{variant}_exercise", len(SUPPORTED_CLASSES), exercise_loaders, SEED + 1, config["input_features"])
    results[variant] = {"config": config, "gate_loaders": gate_loaders, "exercise_loaders": exercise_loaders, "gate_model": gate_model, "exercise_model": exercise_model, "history": gate_history + exercise_history, "best_validation": {"gate": gate_best, "exercise": exercise_best}}


## Validation-only threshold selection per variant

Each threshold is selected on the unchanged validation partition. Test labels
do not influence threshold selection.


In [ ]:
for variant, result in results.items():
    _, _, gate_val_probability = predict_model(result["gate_model"], result["gate_loaders"][1])
    _, exercise_val_prediction, _ = predict_model(result["exercise_model"], result["exercise_loaders"][1])
    # Exercise validation excludes unknown, so score all validation windows directly below.
    features = np.stack([pose_to_features(X[index], result["config"]["mode"]) for index in val_idx]).transpose(0, 2, 1)
    exercise_parts = []
    result["exercise_model"].eval()
    with torch.no_grad():
        for start in range(0, len(features), 256):
            exercise_parts.append(result["exercise_model"](torch.from_numpy(features[start:start + 256]).to(DEVICE)).softmax(1).cpu().numpy())
    exercise_val_prob = np.concatenate(exercise_parts)
    rows = []
    for threshold in np.linspace(0.05, 0.95, 37):
        pred = np.asarray([SUPPORTED_CLASSES[c] if p >= threshold else "unknown" for p, c in zip(gate_val_probability[:, 1], exercise_val_prob.argmax(1))])
        supported = labels[val_idx] != "unknown"; unknown = ~supported
        rows.append({"variant": variant, "threshold": float(threshold), "validation_macro_f1": float(f1_score(labels[val_idx], pred, labels=ALL_CLASSES, average="macro", zero_division=0)), "supported_accuracy": float((pred[supported] == labels[val_idx][supported]).mean()), "unknown_rejection": float((pred[unknown] == "unknown").mean())})
    sweep = pd.DataFrame(rows); best = sweep.sort_values(["validation_macro_f1", "supported_accuracy"], ascending=False).iloc[0]
    result["threshold_sweep"] = sweep; result["threshold"] = float(best.threshold)
    print(variant, "threshold", result["threshold"], "validation F1", float(best.validation_macro_f1))


## Unchanged test report and export

Both variants are compared with original V6-C and Notebook 24 on identical
window keys, with aggregate, per-class, and creator-level evidence.


In [ ]:
REPORT_ROOT = Path("/content/calorie_v6d_visibility_missingness_ablation_report")
if REPORT_ROOT.exists(): shutil.rmtree(REPORT_ROOT)
REPORT_ROOT.mkdir(); shutil.copy2(AUGMENTATION_PREVIEW_PATH, REPORT_ROOT / "missingness_augmentation_preview.png")
all_metrics, all_history, all_sweeps, creator_rows = {}, [], [], []
data_only_by_key = dict(zip(data_only_predictions.window_key, data_only_predictions.predicted_class))
original_by_key = dict(zip(baseline_predictions.window_key, baseline_predictions.predicted_class))
supported_test_mask = labels[test_idx] != "unknown"

for variant, result in results.items():
    gate_truth, gate_pred, gate_prob = predict_model(result["gate_model"], result["gate_loaders"][2])
    exercise_truth, exercise_pred_supported, exercise_prob_supported = predict_model(result["exercise_model"], result["exercise_loaders"][2])
    # Exercise probabilities for every test window.
    features = np.stack([pose_to_features(X[index], result["config"]["mode"]) for index in test_idx]).transpose(0, 2, 1)
    exercise_parts = []; result["exercise_model"].eval()
    with torch.no_grad():
        for start in range(0, len(features), 256): exercise_parts.append(result["exercise_model"](torch.from_numpy(features[start:start + 256]).to(DEVICE)).softmax(1).cpu().numpy())
    exercise_prob = np.concatenate(exercise_parts); exercise_pred = exercise_prob.argmax(1)
    end_pred = np.asarray([SUPPORTED_CLASSES[c] if p >= result["threshold"] else "unknown" for p, c in zip(gate_prob[:, 1], exercise_pred)])
    classifier_truth = labels[test_idx][supported_test_mask]
    classifier_pred = np.asarray([SUPPORTED_CLASSES[c] for c in exercise_pred[supported_test_mask]])
    metric = {
        "input_features": result["config"]["input_features"], "realistic_occlusion": result["config"]["realistic_occlusion"], "gate_threshold": result["threshold"], "best_validation": result["best_validation"],
        "gate": {"accuracy": float(accuracy_score(y_gate[test_idx], gate_pred)), "balanced_accuracy": float(balanced_accuracy_score(y_gate[test_idx], gate_pred)), "macro_f1": float(f1_score(y_gate[test_idx], gate_pred, average="macro", zero_division=0))},
        "exercise_classifier_oracle": {"accuracy": float(accuracy_score(classifier_truth, classifier_pred)), "balanced_accuracy": float(balanced_accuracy_score(classifier_truth, classifier_pred)), "macro_f1": float(f1_score(classifier_truth, classifier_pred, labels=SUPPORTED_CLASSES, average="macro", zero_division=0))},
        "end_to_end": {"accuracy": float(accuracy_score(labels[test_idx], end_pred)), "balanced_accuracy": float(balanced_accuracy_score(labels[test_idx], end_pred)), "macro_f1": float(f1_score(labels[test_idx], end_pred, labels=ALL_CLASSES, average="macro", zero_division=0))},
    }
    for control_name, control in [("original_v6c", baseline_metrics), ("notebook24_data_only", data_only_metrics)]:
        metric[f"delta_vs_{control_name}"] = {"gate_macro_f1": metric["gate"]["macro_f1"] - control["gate"]["macro_f1"], "exercise_oracle_macro_f1": metric["exercise_classifier_oracle"]["macro_f1"] - control["exercise_classifier_oracle"]["macro_f1"], "end_to_end_macro_f1": metric["end_to_end"]["macro_f1"] - control["end_to_end"]["macro_f1"], "end_to_end_accuracy": metric["end_to_end"]["accuracy"] - control["end_to_end"]["accuracy"]}
    all_metrics[variant] = metric; all_history += result["history"]; all_sweeps.append(result["threshold_sweep"])
    reports = {
        f"{variant}_gate_classification_report.json": classification_report(y_gate[test_idx], gate_pred, labels=[0,1], target_names=["unknown","supported"], output_dict=True, zero_division=0),
        f"{variant}_exercise_classification_report.json": classification_report(classifier_truth, classifier_pred, labels=SUPPORTED_CLASSES, output_dict=True, zero_division=0),
        f"{variant}_end_to_end_classification_report.json": classification_report(labels[test_idx], end_pred, labels=ALL_CLASSES, output_dict=True, zero_division=0),
    }
    for filename, report in reports.items(): (REPORT_ROOT / filename).write_text(json.dumps(report, indent=2) + "\n")
    plots = [
        ("gate", y_gate[test_idx], gate_pred, [0, 1], ["unknown", "supported"]),
        ("exercise", classifier_truth, classifier_pred, SUPPORTED_CLASSES, SUPPORTED_CLASSES),
        ("end_to_end", labels[test_idx], end_pred, ALL_CLASSES, ALL_CLASSES),
    ]
    for plot_name, truth, prediction, matrix_labels, tick_labels in plots:
        matrix = confusion_matrix(truth, prediction, labels=matrix_labels)
        plt.figure(figsize=(max(6, len(tick_labels) * 0.75), max(5, len(tick_labels) * 0.65)))
        sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", xticklabels=tick_labels, yticklabels=tick_labels)
        plt.xlabel("Predicted"); plt.ylabel("True"); plt.title(f"{variant} {plot_name}")
        plt.tight_layout(); plt.savefig(REPORT_ROOT / f"{variant}_{plot_name}_confusion_matrix.png", dpi=180); plt.close()
    pd.DataFrame({"window_key": window_keys[test_idx], "group_id": groups[test_idx], "true_class": labels[test_idx], "gate_supported_probability": gate_prob[:,1], "exercise_prediction": [SUPPORTED_CLASSES[c] for c in exercise_pred], "exercise_confidence": exercise_prob.max(axis=1), "predicted_class": end_pred}).to_csv(REPORT_ROOT / f"{variant}_test_window_predictions.csv", index=False)
    for group_id in sorted(set(groups[test_idx])):
        mask = groups[test_idx] == group_id; truth = labels[test_idx][mask]; pred = end_pred[mask]; present = sorted(set(truth))
        controls = {"original": np.asarray([original_by_key[k] for k in window_keys[test_idx][mask]]), "data_only": np.asarray([data_only_by_key[k] for k in window_keys[test_idx][mask]])}
        row = {"variant": variant, "group_id": group_id, "windows": int(mask.sum()), "current_accuracy": float(accuracy_score(truth,pred)), "current_present_macro_f1": float(f1_score(truth,pred,labels=present,average="macro",zero_division=0))}
        for name, cpred in controls.items(): row[f"{name}_accuracy_delta"] = row["current_accuracy"] - accuracy_score(truth,cpred); row[f"{name}_present_macro_f1_delta"] = row["current_present_macro_f1"] - f1_score(truth,cpred,labels=present,average="macro",zero_division=0)
        creator_rows.append(row)
    torch.save({"gate_state_dict": result["gate_model"].state_dict(), "exercise_state_dict": result["exercise_model"].state_dict(), "variant": variant, "classes": ALL_CLASSES, "supported_classes": SUPPORTED_CLASSES, "gate_threshold": result["threshold"], "input_features": result["config"]["input_features"], "feature_config": MASK_FEATURE_CONFIG, "occlusion_config": OCCLUSION_CONFIG if result["config"]["realistic_occlusion"] else None}, REPORT_ROOT / f"two_stage_tcn_{variant}.pt")

report = {"experiment": "v6d_visibility_missingness_ablation", "pinned_source_commit": PINNED_COMMIT, "only_changed_blocks": ["visibility_and_missing_mask_channels", "train_only_realistic_pose_failure_as_separate_variant"], "unchanged_test_windows": len(test_idx), "new_train_groups": sorted(NEW_TRAIN_GROUPS), "base_feature_config": BASE_FEATURE_CONFIG, "mask_feature_config": MASK_FEATURE_CONFIG, "occlusion_config": OCCLUSION_CONFIG, "controls": {"original_v6c": baseline_metrics, "notebook24_data_only": data_only_metrics}, "variants": all_metrics}
(REPORT_ROOT / "metrics.json").write_text(json.dumps(report, indent=2) + "\n")
pd.DataFrame(all_history).to_csv(REPORT_ROOT / "training_history.csv", index=False)
history_frame = pd.DataFrame(all_history)
for model_name, group in history_frame.groupby("model"):
    group.plot(x="epoch", y=["train_macro_f1", "val_macro_f1"], figsize=(8, 4), grid=True, title=model_name)
    plt.tight_layout(); plt.savefig(REPORT_ROOT / f"{model_name}_learning_curve.png", dpi=180); plt.close()
pd.concat(all_sweeps).to_csv(REPORT_ROOT / "gate_threshold_sweeps.csv", index=False)
pd.DataFrame(creator_rows).to_csv(REPORT_ROOT / "test_creator_breakdown.csv", index=False)
visibility_audit.to_csv(REPORT_ROOT / "visibility_source_audit.csv", index=False)
data_delta_audit.to_csv(REPORT_ROOT / "data_delta_audit.csv", index=False)
class_creator_support_matrix.to_csv(REPORT_ROOT / "class_creator_support_matrix.csv")
sampler_audit.to_csv(REPORT_ROOT / "sampler_audit.csv", index=False)
pd.DataFrame({"window_key": window_keys, "group_id": groups, "active_class": labels, "split": assigned_split}).query('split != ""').to_csv(REPORT_ROOT / "split_assignments.csv", index=False)
source_paths = zip_paths + [HARD_POSITIVE_PATH, V6_SPLIT_PATH, BASELINE_METRICS_PATH, BASELINE_PREDICTIONS_PATH, DATA_ONLY_METRICS_PATH, DATA_ONLY_PREDICTIONS_PATH]
(REPORT_ROOT / "source_hashes.json").write_text(json.dumps({p.name: sha256(p) for p in source_paths}, indent=2) + "\n")
checksum_lines = [f"{sha256(p)}  {p.relative_to(REPORT_ROOT)}" for p in sorted(x for x in REPORT_ROOT.rglob('*') if x.is_file())]
(REPORT_ROOT / "SHA256SUMS.txt").write_text("\n".join(checksum_lines) + "\n")
zip_path = shutil.make_archive("/content/calorie_v6d_visibility_missingness_ablation_report", "zip", root_dir=REPORT_ROOT)
print(json.dumps(all_metrics, indent=2)); print("Report ZIP:", zip_path); files.download(zip_path)


## Return the report

Send back `calorie_v6d_visibility_missingness_ablation_report.zip` before the
complete-action temporal experiment is prepared.
